# 00 - Modelo de Churn Passo a Passo Para o Professor

Este notebook ? uma vers?o **simples, did?tica e apresent?vel** do modelo de churn da Aurora Finance Intelligence.

Ele foi feito para qualquer integrante conseguir explicar ao professor:

- o que ? churn;
- como a base ? carregada;
- como limpamos e preparamos os dados;
- como separamos treino e teste;
- como treinamos modelos simples;
- como avaliamos os resultados;
- como geramos uma lista final de clientes priorizados.

Importante: este notebook ? uma explica??o passo a passo. O pipeline oficial e mais estruturado do projeto continua em `src/`.

## 1. Importa??o das bibliotecas

Vamos usar bibliotecas comuns em cursos de dados:

- `pandas` para tabelas;
- `numpy` para c?lculos num?ricos;
- `matplotlib` e `seaborn` para gr?ficos;
- `sklearn` para Machine Learning.

Se `seaborn` n?o estiver instalado, o notebook continua funcionando com `matplotlib`.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
    SEABORN_DISPONIVEL = True
except ImportError:
    sns = None
    SEABORN_DISPONIVEL = False
    print("Seaborn n?o est? instalado. O notebook seguir? usando matplotlib.")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_PATH = ROOT / "dados" / "raw" / "churn_modelling.csv"
CLIENTES_PROCESSADOS_PATH = ROOT / "dados" / "processed" / "clientes_limpo.csv"
OUTPUT_PATH = ROOT / "dados" / "outputs" / "predicoes_churn_notebook_didatico.csv"

print(f"Raiz do projeto: {ROOT}")
print(f"Seaborn dispon?vel: {SEABORN_DISPONIVEL}")

## 2. Leitura da base

A base principal ? o **Churn Modelling**, um dataset p?blico do Kaggle sobre clientes banc?rios.

O arquivo esperado ?:

`dados/raw/churn_modelling.csv`

Se ele n?o existir, usamos `dados/processed/clientes_limpo.csv`, que ? gerado pelo pipeline oficial:

```powershell
python -m src.pipeline
```

In [ ]:
if RAW_PATH.exists():
    df = pd.read_csv(RAW_PATH)
    fonte = "raw_kaggle"
    print("Base p?blica Churn Modelling carregada.")
elif CLIENTES_PROCESSADOS_PATH.exists():
    df_processado = pd.read_csv(CLIENTES_PROCESSADOS_PATH)
    fonte = "clientes_processados"
    print("CSV p?blico n?o encontrado. Usando clientes_limpo.csv gerado pelo pipeline.")
    print("Se quiser usar o Kaggle, salve o arquivo em dados/raw/churn_modelling.csv e rode python -m src.pipeline.")
else:
    raise FileNotFoundError(
        "Nenhuma base encontrada. Rode primeiro: python -m src.pipeline"
    )

print("Fonte usada:", fonte)
print("Dimens?es:", df.shape if fonte == "raw_kaggle" else df_processado.shape)

## 3. Explica??o do dataset

No `Churn Modelling`, cada linha representa um cliente. Algumas colunas importantes:

- `Exited`: target de churn. Vale `1` quando o cliente saiu e `0` quando permaneceu.
- `EstimatedSalary`: sal?rio anual estimado.
- `Balance`: saldo do cliente.
- `CreditScore`: score de cr?dito.
- `Geography`: localiza??o do cliente.
- `NumOfProducts`: n?mero de produtos contratados.
- `IsActiveMember`: indica se o cliente ? ativo.

Em um problema de churn, queremos prever quais clientes t?m maior chance de sair para priorizar a??es de reten??o.

## 4. Adapta??o quando usamos a base processada

Se estivermos usando `clientes_limpo.csv`, ele j? vem no schema Aurora, em portugu?s. Para manter o notebook simples, criamos nomes equivalentes aos do Kaggle.

In [ ]:
if fonte == "clientes_processados":
    df = pd.DataFrame({
        "CustomerId": df_processado.get("customer_id_original", df_processado["cliente_id"]),
        "Surname": df_processado.get("nome", "Cliente"),
        "CreditScore": df_processado["score_credito"],
        "Geography": df_processado["estado"],
        "Gender": df_processado["genero"],
        "Age": df_processado["idade"],
        "Tenure": df_processado.get("tempo_relacionamento", 0),
        "Balance": df_processado["saldo_atual"],
        "NumOfProducts": df_processado["produtos_ativos"],
        "HasCrCard": df_processado.get("tem_cartao_credito", 0),
        "IsActiveMember": df_processado["membro_ativo"],
        "EstimatedSalary": df_processado["renda_mensal"] * 12,
        "Exited": df_processado["churn_flag"],
    })
else:
    df_processado = None

display(df.head())

## 5. An?lise inicial da base

Antes de modelar, precisamos conhecer a base:

- `head()` mostra as primeiras linhas;
- `info()` mostra tipos e nulos;
- `describe()` mostra estat?sticas;
- `shape` mostra quantidade de linhas e colunas;
- `isnull().sum()` mostra nulos por coluna;
- `duplicated().sum()` mostra linhas duplicadas.

In [ ]:
display(df.head())
print("Shape:", df.shape)

In [ ]:
df.info()

In [ ]:
display(df.describe(include="all"))

In [ ]:
print("Nulos por coluna:")
display(df.isnull().sum())
print("Linhas duplicadas:", df.duplicated().sum())

## 6. O que ? churn?

Churn significa sa?da, cancelamento ou abandono de relacionamento.

Neste dataset:

- `Exited = 0`: cliente n?o saiu;
- `Exited = 1`: cliente saiu.

Nosso objetivo ? aprender padr?es que indiquem maior chance de `Exited = 1`.

In [ ]:
taxa_churn = df["Exited"].mean()
print(f"Taxa de churn da base: {taxa_churn:.2%}")

contagem_churn = df["Exited"].value_counts().sort_index()
plt.figure(figsize=(5, 4))
plt.bar(["N?o saiu", "Saiu"], contagem_churn.values, color=["#20c7df", "#ff6b7a"])
plt.title("Distribui??o do churn")
plt.ylabel("Quantidade de clientes")
plt.show()

## 7. Limpeza e prepara??o

Para treinar um modelo simples, removemos colunas que n?o ajudam diretamente ou que s?o identificadores:

- `RowNumber`: n?mero da linha;
- `CustomerId`: identificador do cliente;
- `Surname`: sobrenome/nome do cliente.

Depois transformamos vari?veis categ?ricas, como `Gender` e `Geography`, em colunas num?ricas com `pd.get_dummies()`.

In [ ]:
colunas_remover = ["RowNumber", "CustomerId", "Surname"]
dados_modelo = df.drop(columns=[c for c in colunas_remover if c in df.columns]).copy()

# Separar target antes de transformar as features.
y = dados_modelo["Exited"].astype(int)
X = dados_modelo.drop(columns=["Exited"])

X = pd.get_dummies(X, columns=["Gender", "Geography"], drop_first=True, dtype=int)

print("Features finais:", X.shape)
display(X.head())

## 8. Treino e teste

Dividimos a base em duas partes:

- **Treino:** usada para o modelo aprender padr?es.
- **Teste:** usada para avaliar se o modelo generaliza para clientes que ele ainda n?o viu.

Usamos `stratify=y` para manter a mesma propor??o de churn no treino e no teste.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)
print("Churn no treino:", y_train.mean().round(4))
print("Churn no teste:", y_test.mean().round(4))

## 9. Treinamento de modelos simples

Vamos comparar tr?s modelos:

1. `LogisticRegression`: baseline simples, muito comum em cursos.
2. `DecisionTreeClassifier`: ?rvore de decis?o, mais interpret?vel.
3. `RandomForestClassifier`: evolu??o com v?rias ?rvores, mais robusta.

A Random Forest ? o modelo mais pr?ximo do pipeline oficial da Aurora.

In [ ]:
modelos = {
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    ]),
    "DecisionTree": DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1),
}

modelos_treinados = {}
resultados = []

for nome, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    modelos_treinados[nome] = modelo
    prob = modelo.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    resultados.append({
        "modelo": nome,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1_score": f1_score(y_test, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, prob),
    })

resultados_df = pd.DataFrame(resultados).sort_values("roc_auc", ascending=False)
display(resultados_df)

## 10. Como interpretar as m?tricas?

- `accuracy`: percentual geral de acertos.
- `precision`: dos clientes marcados como risco, quantos realmente sa?ram.
- `recall`: dos clientes que realmente sa?ram, quantos o modelo conseguiu capturar.
- `f1-score`: equil?brio entre precision e recall.
- `roc_auc`: capacidade de ranquear clientes de maior e menor risco.

Em churn, **recall importa muito**, porque falso negativo significa deixar passar um cliente que poderia sair.

Mas **precision tamb?m importa**, porque muitos falsos positivos podem gerar uma fila de reten??o grande demais ou a??es desnecess?rias.

## 11. Falso positivo e falso negativo

Para churn:

- **Falso positivo:** o modelo diz que o cliente vai sair, mas ele n?o sairia. A empresa pode gastar esfor?o de reten??o sem necessidade.
- **Falso negativo:** o modelo diz que o cliente n?o vai sair, mas ele sai. A empresa perde a chance de agir antes.

Por isso, o modelo n?o deve decidir sozinho. Ele deve **priorizar clientes** para uma avalia??o humana.

In [ ]:
modelo_final_nome = "RandomForest"
modelo_final = modelos_treinados[modelo_final_nome]
prob_final = modelo_final.predict_proba(X_test)[:, 1]
pred_final = (prob_final >= 0.5).astype(int)

print(f"Classification report - {modelo_final_nome}")
print(classification_report(y_test, pred_final, target_names=["N?o churn", "Churn"]))

## 12. Matriz de confus?o

A matriz de confus?o mostra os acertos e erros do modelo.

Ela ajuda a enxergar falsos positivos e falsos negativos de forma visual.

In [ ]:
cm = confusion_matrix(y_test, pred_final)

plt.figure(figsize=(5, 4))
if SEABORN_DISPONIVEL:
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["N?o churn", "Churn"], yticklabels=["N?o churn", "Churn"])
else:
    plt.imshow(cm, cmap="Blues")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j], ha="center", va="center", color="black")
    plt.xticks([0, 1], ["N?o churn", "Churn"])
    plt.yticks([0, 1], ["N?o churn", "Churn"])
plt.title("Matriz de confus?o")
plt.xlabel("Predito")
plt.ylabel("Real")
plt.show()

## 13. Feature importance

A feature importance mostra quais vari?veis mais ajudaram o modelo a separar clientes com maior e menor risco de churn.

Isso n?o prova causalidade, mas ajuda a explicar o comportamento do modelo.

In [ ]:
if hasattr(modelo_final, "feature_importances_"):
    importancias = pd.DataFrame({
        "feature": X.columns,
        "importance": modelo_final.feature_importances_,
    }).sort_values("importance", ascending=False)
else:
    importancias = pd.DataFrame({
        "feature": X.columns,
        "importance": np.abs(modelo_final.named_steps["model"].coef_[0]),
    }).sort_values("importance", ascending=False)

display(importancias.head(10))

plt.figure(figsize=(8, 5))
top_importancias = importancias.head(10).sort_values("importance")
plt.barh(top_importancias["feature"], top_importancias["importance"], color="#20c7df")
plt.title("Top 10 vari?veis mais importantes")
plt.xlabel("Import?ncia")
plt.show()

## 14. Gerando uma base final de prioriza??o

Agora criamos uma tabela simples para simular a sa?da do modelo:

- `cliente_id`: identificador do cliente;
- `prob_churn`: probabilidade estimada de churn;
- `risco`: classifica??o em baixo, m?dio ou alto;
- `recomendacao`: a??o sugerida para a equipe.

Essa tabela ? did?tica. O pipeline oficial j? gera uma vers?o mais completa em `dados/outputs/predicoes_churn.csv`.

In [ ]:
prob_todos = modelo_final.predict_proba(X)[:, 1]

def classificar_risco(prob):
    if prob >= 0.60:
        return "Alto"
    if prob >= 0.35:
        return "Medio"
    return "Baixo"

def recomendar(risco):
    if risco == "Alto":
        return "Priorizar contato consultivo e oferta de reten??o personalizada."
    if risco == "Medio":
        return "Monitorar engajamento e sugerir benef?cio segmentado."
    return "Manter relacionamento ativo e acompanhar evolu??o."

if "CustomerId" in df.columns:
    cliente_id = df["CustomerId"].astype(str)
else:
    cliente_id = pd.Series([f"C{i:06d}" for i in range(1, len(df) + 1)])

predicoes_didaticas = pd.DataFrame({
    "cliente_id": cliente_id,
    "prob_churn": prob_todos,
})
predicoes_didaticas["risco"] = predicoes_didaticas["prob_churn"].apply(classificar_risco)
predicoes_didaticas["recomendacao"] = predicoes_didaticas["risco"].apply(recomendar)

predicoes_didaticas = predicoes_didaticas.sort_values("prob_churn", ascending=False)
display(predicoes_didaticas.head(20))

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
predicoes_didaticas.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(f"Arquivo salvo em: {OUTPUT_PATH}")

## 15. Conclus?o

Este notebook representa o passo a passo did?tico pedido no curso:

1. carregamos a base;
2. entendemos o dataset;
3. fizemos an?lise inicial;
4. limpamos e preparamos as vari?veis;
5. separamos treino e teste;
6. treinamos modelos simples;
7. avaliamos m?tricas;
8. interpretamos matriz de confus?o e feature importance;
9. geramos uma lista final de clientes priorizados.

O pipeline em `src/` automatiza esse mesmo racioc?nio em uma vers?o mais estruturada, reprodut?vel e pronta para produ??o local.

O app React e a An?lise Expressa s?o diferenciais visuais do projeto. Eles n?o substituem o notebook, o SQL, o Power BI nem o modelo oficial.